# **Dataset Analysis** 

## **Colums**

In [20]:
#pip install pandas

In [21]:
import pandas as pd  

# Read the CSV file into memory
data = pd.read_csv("Medicine_Details.csv") 

In [22]:
# Display the first few rows of the dataframe
data.head()

,Medicine Name,Composition,Uses,Side_effects,Image URL,Manufacturer,Excellent Review %,Average Review %,Poor Review %
0,Avastin 400mg Injection,Bevacizumab (400mg),Cancer of colon and rectum Non-small cell lun...,Rectal bleeding Taste change Headache Noseblee...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Roche Products India Pvt Ltd,22,56,22
1,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),Treatment of Bacterial infections,Vomiting Nausea Diarrhea Mucocutaneous candidi...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glaxo SmithKline Pharmaceuticals Ltd,47,35,18
2,Azithral 500 Tablet,Azithromycin (500mg),Treatment of Bacterial infections,Nausea Abdominal pain Diarrhea,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Alembic Pharmaceuticals Ltd,39,40,21
3,Ascoril LS Syrup,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml)...,Treatment of Cough with mucus,Nausea Vomiting Diarrhea Upset stomach Stomach...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glenmark Pharmaceuticals Ltd,24,41,35
4,Aciloc 150 Tablet,Ranitidine (150mg),Treatment of Gastroesophageal reflux disease (...,Headache Diarrhea Gastrointestinal disturbance,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Cadila Pharmaceuticals Ltd,34,37,29


## **Missing values**

In [23]:
# Check for missing values
missing_values = data.isnull().sum()

missing_values

Medicine Name         0
Composition           0
Uses                  0
Side_effects          0
Image URL             0
Manufacturer          0
Excellent Review %    0
Average Review %      0
Poor Review %         0
dtype: int64

# **Data Preprocessing** 

## **1. Converting the original dataset (.csv) into a structured JSON format.**

### **1.1. *Verification: Is Abnormal always followed by E C G when it has a capital letter after it in side_effects?***

In [ ]:
import re
import pandas as pd

df = data.copy()

print(f"Analysis of {len(df)} filtered medicines")

# check if Abnormal is always followed by E C G when a capital letter comes right after it in side_effects
print("\nPhase 1: 'Abnormal' + uppercase analysis")

found_cases = []

# iterate the dataframe to get the medicine name and the row
for index, row in df.iterrows():
    text = str(row['Side_effects']).strip()
    
    # Regex: Abnormal + space + capital letter
    matches = re.findall(r'(Abnormal\s+[A-Z]\w*)', text)
    
    if matches:
        for match in matches:
            found_cases.append({
                "Row": index + 2,
                "Medicine": row.get('Medicine Name', 'N/A'),
                "Pattern": match
            })

if len(found_cases) > 0:
    results_df = pd.DataFrame(found_cases)
    pd.set_option('display.max_rows', None)
    print(f"Found {len(found_cases)} cases.")
    #print(results_df[['Medicine', 'Pattern']])
else:
    print("No cases of 'Abnormal' followed by an uppercase letter were found.")

# check connectors followed by an uppercase letter that actually belong to "Uses"/"Side_effects"
print("\nPhase 2: connector validation")

# put all the text into one list (Uses + Side_effects) 
all_text_data = pd.concat([df['Uses'], df['Side_effects']]).dropna().astype(str).tolist()

connectors = {
    'of', 'with', 'in', 'to', 'for', 'and', '&', 
    'associated', 'related', 'caused', 'by', 'due', 'from',
    'during', 'after', 'before'
}

print(f"\n{'CONNECTOR':<15} | {'CAPITAL LETTER AFTER?':<22} | {'EXAMPLE FOUND'}")

for conn in sorted(connectors): 
    found = False
    example = ""
    
    # correct regex (with or without \b)
    if re.match(r'^\w+$', conn):
        # it's a word (e.g. "and") -> use \b to avoid matching "and" inside "hand"
        pattern = r'(?i)\b' + re.escape(conn) + r'\b\s+(\w+)'
    else:
        # it's a symbol (e.g. "&") -> don't use \b
        pattern = r'(?i)' + re.escape(conn) + r'\s+(\w+)'
    
    # go through all the text
    for text in all_text_data:
        matches = re.finditer(pattern, text)
        
        for match in matches:
            # the word right after it
            next_word = match.group(1)
            
            # check if the first letter is really uppercase via .isupper()
            # this ignores the (?i) flag effect
            if next_word[0].isupper():
                found = True
                example = match.group(0) # store the whole match (e.g. "by Virus")
                break 
        
        if found: break 
    
    status = "YES" if found else "NO"
    print(f"{conn:<15} | {status:<22} | {example}")

Analysis of 11825 filtered medicines

Phase 1: 'Abnormal' + uppercase analysis
Found 17 cases.

Phase 2: connector validation

CONNECTOR       | CAPITAL LETTER AFTER?  | EXAMPLE FOUND
&               | NO                     | 
after           | NO                     | 
and             | NO                     | 
associated      | NO                     | 
before          | NO                     | 
by              | NO                     | 
caused          | NO                     | 
due             | NO                     | 
during          | NO                     | 
for             | NO                     | 
from            | NO                     | 
in              | YES                    | in Parkinson
of              | YES                    | of Bacterial
related         | NO                     | 
to              | YES                    | to ITP
with            | YES                    | with Huntington


### **1.2 Conversion**

1. **Filtering:** Removal of irrelevant columns (`Manufacturer`, `Reviews`, `URL`).
2. **Text Segmentation:** `Uses`, `Side_effects` - Separation of concatenated strings based on **Uppercase Letters**.
* **Rule:** Uppercase indicates a new item.
* **Exception:** If preceded by connectors (`of`, `by`, `with`) or protected by a mask.
3. **Text Segmentation:** `Composition` - Separation of concatenated strings based on "+".
4. **Normalization:**
* Merging of acronyms (`E C G` $\to$ `ECG`).
* Handling redundancies (`Treatment of Prevention of`).
* **Edge Case:** Specific protection against `Abnormal ECG` via *masking* (`Abnormal_ECG`). As well as "Vitamin B12 deficiency", "herpes simplex virus infection", "treatment of vitamin A deficiency", among others.

In [25]:
import pandas as pd
import json
import re
import os

# Text cleaning function (Uses and side effects)
def clean_medical_text(text):
    """
    Cleans concatenated medical text.
    Fixes redundant prefixes, joins acronyms, protects compound medical terms, 
    and dynamically splits text based on Capitalization rules.
    """
    if pd.isna(text) or text == "": 
        return []
    
    text = str(text).strip()
    
    # 1. PREFIX CORRECTIONS & ACRONYMS
    # Convert descriptive conversational phrases into standard prefixes
    text = re.sub(r'(?i)It is used to treat\s+', 'Treatment of ', text)
    # Remove redundant "Treatment of" if followed by "Prevention of"
    text = re.sub(r'Treatment of\s+(?=Prevention of)', '', text, flags=re.IGNORECASE) #remove Treatment of

    # Join separated acronyms: "E C G" -> "ECG"
    text = re.sub(r'(\b[A-Z])\s+(?=[A-Z]\b)', r'\1', text)
    text = re.sub(r'(\b[A-Z])\s+(?=[A-Z]\b)', r'\1', text)

    # 2. COMPOUND EXCEPTIONS (Masking spaces with underscores)
    # This prevents the Capitalization Loop from splitting these specific terms
    
    # --- Diagnostics & General Conditions ---
    text = re.sub(r'(?i)Abnormal\s+ECG', 'Abnormal_ECG', text)
    text = re.sub(r'(?i)Severe\s+Aplastic', 'Severe_Aplastic', text)
    text = re.sub(r'(?i)Acute\s+Sore\s+throat', 'Acute_Sore_throat', text)
    text = re.sub(r'(?i)Improving\s+Abdominal\s+pain', 'Improving_Abdominal_pain', text)
    text = re.sub(r'(?i)Dry\s+Skin', 'Dry_Skin', text)
    # Protect specific longer phrases FIRST
    text = re.sub(r'(?i)Burning\s+sensation\s+in\s+Eye\s+Irritation', 'Burning_sensation_in_Eye ||| Irritation', text) #Proflur Eye Drop
    text = re.sub(r'(?i)Eye\s+Irritation', 'Eye_Irritation', text) #this one was causing the following case: "Burning sensation in Eye Irritation"
    text = re.sub(r'(?i)Breast\s+Disease', 'Breast_Disease', text)
    text = re.sub(r'(?i)Cervical\s+Dystonia', 'Cervical_Dystonia', text)
    text = re.sub(r'(?i)Leber.?s\s+Hereditary\s+Optic\s+Neuropathy', "Leber's_Hereditary_Optic_Neuropathy", text)
    
    # --- Infections, Viruses & Bacteria ---
    text = re.sub(r'(?i)Herpes\s+Simplex\s+Virus\s+Infections?', 'Herpes_Simplex_Virus_Infections', text)
    text = re.sub(r'(?i)Cytomegalovirus\s+Infection', 'Cytomegalovirus_Infection', text)
    text = re.sub(r'(?i)Japanese\s+Encephalitis', 'Japanese_Encephalitis', text)
    text = re.sub(r'(?i)Haemophilus\s+influenzae\s+Type\s+b', 'Haemophilus_influenzae_Type_b', text)
    
    # Tuberculosis (Resistance and Resistant)
    text = re.sub(r'(?i)Resistance\s+Tuberculosis\s*\(\s*TB\s*\)', 'Resistance_Tuberculosis_(TB)', text)
    text = re.sub(r'(?i)Resistant\s+Tuberculosis\s*\(\s*TB\s*\)', 'Resistant_Tuberculosis_(TB)', text)
    
    # Hepatitis (Catches A, B, or C automatically)
    text = re.sub(r'(?i)hepatitis\s+([ABC])', r'hepatitis_\1', text)
    text = re.sub(r'(?i)Non-Alcoholic\s+Steatohepatitis', 'Non-Alcoholic_Steatohepatitis', text)

    # --- Vitamins & Deficiencies ---
    # Dynamically catches Vitamin B12, C, D, A, E deficiency, etc.
    text = re.sub(r'(?i)Vitamin\s+([A-Z0-9]+)\s+deficiency', r'Vitamin_\1_deficiency', text)
    
    # --- Procedures & External Events ---
    text = re.sub(r'(?i)General\s+Anaesthesia', 'General_Anaesthesia', text)
    text = re.sub(r'(?i)Premature\s+Labor', 'Premature_Labor', text)
    text = re.sub(r'(?i)Snake\s+Bite', 'Snake_Bite', text)

    # 3. INSERT DELIMITERS FOR PREFIXES
    start_keywords = ["Treatment of", "Prevention of", "Treatment and prevention of"] 
    pattern_prefix = r'(' + '|'.join([re.escape(p) for p in start_keywords]) + r')'
    text = re.sub(pattern_prefix, r'||| \1', text)
    
    raw_chunks = [t.strip() for t in text.split('|||') if t.strip()] #"Treatment of Heart Attack" ||| "Prevention of stroke"
    final_items = []

    # 4. CAPITALIZATION SPLIT LOOP
    # If a word starts with a Capital letter, it's a new symptom unless preceded by a connector
    connectors = {'of', 'with', 'in', 'to', 'and'}

    for chunk in raw_chunks:
        words = chunk.split(' ')
        current_phrase = []
        
        for i, word in enumerate(words): 
            if i == 0:
                current_phrase.append(word)
                continue
            
            clean_word = word.strip()
            if not clean_word: continue # if it were " " (empty), move on to the next word

            # Only split if it is Capitalized AND the previous word is NOT a connector
            is_capitalized = clean_word[0].isupper()
            prev_word = words[i-1].lower().strip()
            
            if is_capitalized and (prev_word not in connectors):
                final_items.append(" ".join(current_phrase))
                current_phrase = [word]
            else:
                current_phrase.append(word) 
        
        if current_phrase:
            final_items.append(" ".join(current_phrase))

    # 5. FINAL CLEANUP & RESTORATION
    prefixes_lower = [p.lower() for p in start_keywords]
    cleaned_final = []
    
    for item in final_items:
        # Restore the spaces for all masked compound terms automatically
        item = item.replace('_', ' ') 

        item = item.rstrip('.').strip()
        
        if item.lower() not in prefixes_lower and len(item) > 2: #minimum of 3 characters
            cleaned_final.append(item)

    return cleaned_final

# Helper function (composition)
def clean_composition(text):
    if pd.isna(text) or text == "": return []
    return [ingredient.strip() for ingredient in str(text).split('+')]

# Execution
output_file = "processed_medicines.json"

if 'data' not in locals():
    print("Error: the 'data' variable was not found in memory.")
else:
    try:
        json_df = data.copy()

        print("Processing textual data (Uses, Side_effects, Composition)...")
        json_df['Uses'] = json_df['Uses'].apply(clean_medical_text)
        json_df['Side_effects'] = json_df['Side_effects'].apply(clean_medical_text)
        json_df['Composition'] = json_df['Composition'].apply(clean_composition)

        # Remove irrelevant columns
        cols_to_remove = [
            'Excellent Review %', 
            'Average Review %', 
            'Poor Review %', 
            'Manufacturer', 
            'Image URL'  
        ]
        
        cols_existing = [c for c in cols_to_remove if c in json_df.columns]
        
        if cols_existing:
            json_df.drop(columns=cols_existing, inplace=True)
            print(f"Removed columns: {cols_existing}")
        else:
            print("No irrelevant columns found for removal.")

        # Export to json
        json_data = json_df.to_dict(orient='records')
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)

        print(f"Final dataset saved to '{output_file}'")

    except Exception as e:
        print(f"Error: {e}")

Processing textual data (Uses, Side_effects, Composition)...
Removed columns: ['Excellent Review %', 'Average Review %', 'Poor Review %', 'Manufacturer', 'Image URL']
Final dataset saved to 'processed_medicines.json'


### **1.3 *Check for single words in uses***

In [26]:
import json
import os

INPUT_FILE = "processed_medicines.json"
OUTPUT_FILE = "single_word_uses.json"

def extract_single_word_uses():
    print("Extracting single-word uses...")
    
    # 1. Check if input file exists
    if not os.path.exists(INPUT_FILE):
        print(f"Error: file not found: {INPUT_FILE}")
        return
        
    # 2. Load the processed dataset
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    single_word_set = set()
    
    # 3. Iterate and find single words
    for item in data:
        uses = item.get('Uses', [])
        for use in uses:
            use_clean = use.strip()
            
            # If splitting by space results in exactly 1 chunk, it's a single word
            if len(use_clean.split()) == 1:
                single_word_set.add(use_clean)
                
    # 4. Sort alphabetically for easier manual review
    single_words_list = sorted(list(single_word_set))
    
    # 5. Export to a new JSON file
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(single_words_list, f, indent=4, ensure_ascii=False)
        
    # 6. Print the summary report
    print(f"Found {len(single_words_list)} unique single-word uses.")
    print(f"Saved to '{OUTPUT_FILE}'.")

if __name__ == "__main__":
    extract_single_word_uses()

Extracting single-word uses...
Found 70 unique single-word uses.
Saved to 'single_word_uses.json'.


## **2. Removal of patient-directed information**

The JSON contained information with an instructional and informal tone typical of patient information leaflets (you/yours), e.g., "You need to take it regularly as it is prescribed for it to be most effective and should not stop taking it suddenly." These lines have been removed.

In [ ]:
import json
import re

input_file = "processed_medicines.json"      
output_file = "processed_medicines.json"


try:
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # regex to detect "you", "your"
    regex_you = re.compile(r'\b(you|your)\b', re.IGNORECASE)

    changed_medicines = []
    emptied_medicines = []


    for entry in data:
        med_name = entry.get('Medicine Name', 'Unknown')
        original_uses = entry.get('Uses', [])
        
        new_uses = []
        removed_lines = []

        for phrase in original_uses:
            # removal criteria:
            has_name = "$name" in phrase
            has_you = regex_you.search(phrase)

            if has_name or has_you:
                removed_lines.append(phrase) 
            else:
                new_uses.append(phrase)       # keep for the final file

        # if there were changes to the medicine's uses
        if len(removed_lines) > 0:
            changed_medicines.append({
                "name": med_name,
                "removed_count": len(removed_lines),
                "examples": removed_lines # save the first example of what got deleted
            })

        # update the JSON
        entry['Uses'] = new_uses

        # check: did it end up empty?
        if len(original_uses) > 0 and len(new_uses) == 0:
            emptied_medicines.append(med_name)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Total medicines changed: {len(changed_medicines)}")
    if changed_medicines:
        for item in changed_medicines:
            print(f"  - {item['name']}: removed {item['removed_count']} lines")

    print(f"\nMedicines that ended up with no 'Uses' (empty): {len(emptied_medicines)}")
    if emptied_medicines:
        for name in emptied_medicines:
            print(f"  {name}")
    else:
        print("  No medicine ended up empty. All of them have at least one use.")

    print(f"\nFile saved to: {output_file}")

except FileNotFoundError:
    print(f"Error: file '{input_file}' does not exist.")
except Exception as e:
    print(f"Unexpected error: {e}")

Total medicines changed: 7
  - DO RE ME 25mg Tablet: removed 3 lines
  - DO RE ME 75mg Tablet: removed 3 lines
  - DO RE ME 50mg Tablet: removed 3 lines
  - Osule 25mg Tablet: removed 3 lines
  - Prothiaden Tablet: removed 3 lines
  - Prothiaden 75 Tablet: removed 3 lines
  - Prothiaden 50 Tablet: removed 3 lines

Medicines that ended up with no 'Uses' (empty): 0
  No medicine ended up empty. All of them have at least one use.

File saved to: processed_medicines.json


## **3. Removing "Limited data available"**

In [ ]:
import json
import os

input_file = "processed_medicines.json"
output_file = "processed_medicines.json"
target_phrase = "Limited data available"

affected_meds = []       # all the ones that had any change
empty_uses_meds = []     # ended up with no Uses
empty_sides_meds = []    # ended up with no Side_effects

print(f"Reading {input_file}...")

if not os.path.exists(input_file):
    print(f"Error: file '{input_file}' does not exist.")
else:
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # --- PROCESSING ---
    for med in data:
        # try to get the name, fall back if missing
        name = med.get('Medicine Name', med.get('Medicine_Name', 'Unknown'))
        
        original_uses = med.get('Uses', [])
        original_sides = med.get('Side_effects', [])
        
        # filter out the unwanted phrase (case insensitive)
        new_uses = [u for u in original_uses if target_phrase.lower() not in u.lower()]
        new_sides = [s for s in original_sides if target_phrase.lower() not in s.lower()]
        
        # check for changes
        uses_changed = len(new_uses) < len(original_uses)
        sides_changed = len(new_sides) < len(original_sides)
        
        if uses_changed or sides_changed:
            affected_meds.append(name)
            
            # update the data in the object
            med['Uses'] = new_uses
            med['Side_effects'] = new_sides
            
            # check if it ended up empty after cleaning
            if uses_changed and len(new_uses) == 0:
                empty_uses_meds.append(name)
            
            if sides_changed and len(new_sides) == 0:
                empty_sides_meds.append(name)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Clean file saved to: {output_file}\n")

    print(f"Removal summary for '{target_phrase}':")

    # total changed
    print(f"\nTotal medicines changed: {len(affected_meds)}")
    if affected_meds:
        print(f"  Names: {affected_meds}")

    # empty Uses
    print(f"\nEnded up with no 'Uses' (empty list): {len(empty_uses_meds)}")
    if empty_uses_meds:
        print(f"  Names: {empty_uses_meds}")
    else:
        print("  (No medicine ended up with no Uses)")

    # empty Side_effects
    print(f"\nEnded up with no 'Side_effects' (empty list): {len(empty_sides_meds)}")
    if empty_sides_meds:
        print(f"  Names: {empty_sides_meds}")
    else:
        print("  (No medicine ended up with no Side_effects)")

Reading processed_medicines.json...
Clean file saved to: processed_medicines.json

Removal summary for 'Limited data available':

Total medicines changed: 98
  Names: ['Aquim PF Eye Drop', 'Amortas Cream', 'Aquaray Plus  Eye Drop', 'Amrobrut Cream', 'Amoron Cream', 'Amrobrut Cream', 'Amortas Cream', 'Amcrylate Bio-Adhesive', 'Aeocal Max Softgel Capsule', 'Amocop Cream', 'Bilagra M 20mg/10mg Tablet', 'Bilazest M Tablet', 'Bilacad M Tablet', 'Bisgram Tablet', 'Biofil-AB Particles', 'Billargic M Tablet', 'Belina Gel', 'Enzoflam-CT Tablet', 'Enzomac CM Particles', 'Eubri Eye Drop', 'Emrolfin Cream', 'Ecofab-HA Eye Drop', 'Fungicros Cream', 'Finetears Eye Drop', 'Forcutril Plus Cream', 'Gabafix Gel', 'Ginlac-V Vaginal Tablet', 'Gastrozin 75mg Tablet', 'Hadensa Ointment', 'Hylosoft Eye Drop', 'Hylosurge Eye Drop', 'Hyla PF Eye Drop', 'Hyvet  Eye Drop', 'Hyane Eye Drop', 'Hyla Fusion Eye Drop', 'Healtears Eye Drop', 'Hyla Eye Drop', 'Hyalgan Injection', 'Hyadry 0.1% Eye Drop', 'Hyal Oral Tabl

---

## **4. Addition of Mechanisms of Action from the DrugBank Database**

The key `Mechanism_of_action` was added in the form of a dictionary (object), where each active ingredient in the composition is explicitly mapped to its respective mechanism of action. If the mechanism of action for a given ingredient does not exist, the value of that ingredient's key will be null.

Both the generic_name and synonyms from the drugbank dataset were considered.

In [29]:
import json
import re
import os

meds_file = "processed_medicines.json"
drugbank_file = os.path.join("..", "DrugBank", "drug_data_test_status.json")
output_file = "processed_medicines_MoA_DrugBank.json"

# load the DrugBank dataset
try:
    with open(drugbank_file, 'r', encoding='utf-8') as f:
        drugbank_list = json.load(f)
except FileNotFoundError:
    print(f"ERROR: file not found: {drugbank_file}")
    drugbank_list = []

# build a dict for generic_name and another for synonyms (assuming synonyms share the same mechanism of action as the generic_name)
generic_lookup = {}   # priority 1
synonym_lookup = {}   # priority 2

for item in drugbank_list:
    mechanism = item.get('Mechanism_of_action', None)
    if not mechanism: continue # skip if there's no mechanism

    # 1. index the generic name
    g_name = item.get('generic_name', '').strip()
    if g_name:
        generic_lookup[g_name.lower()] = mechanism

    # 2. index the synonyms
    synonyms_str = item.get('sinonimos', '')
    if synonyms_str:
        parts = synonyms_str.split(',')
        for s in parts:
            clean_syn = s.strip().lower()
            if clean_syn:
                synonym_lookup[clean_syn] = mechanism 

print(f"Indexing complete.")
print(f"Generics: {len(generic_lookup)}")
print(f"Synonyms: {len(synonym_lookup)}")

# load the previously processed medicines dataset
try:
    with open(meds_file, 'r', encoding='utf-8') as f:
        medicines_data = json.load(f)
except FileNotFoundError:
    print(f"ERROR: file '{meds_file}' not found.")
    medicines_data = []

# merge the data from both datasets

count_enriched = 0

for med in medicines_data:
    composition = med.get('Composition', [])
    
    # dict that stores the result { Ingredient: Mechanism or None }
    mechanisms_dict = {} 
    
    if not isinstance(composition, list):
        composition = []

    for ingredient_raw in composition:
        # clean-up: "Fenofibrate (160mg)" -> "Fenofibrate"
        ingredient_clean = re.sub(r'\s*\([^)]*\)', '', ingredient_raw).strip()
        ingredient_lower = ingredient_clean.lower()
        
        mech_text = None
        
        # 1st try: look it up in the generic names
        if ingredient_lower in generic_lookup:
            mech_text = generic_lookup[ingredient_lower]
        
        # 2nd try: if that failed, check the synonyms
        elif ingredient_lower in synonym_lookup:
            mech_text = synonym_lookup[ingredient_lower]

        # assign the value (text or None) to the dict
        mechanisms_dict[ingredient_clean] = mech_text

    # store the dict on the medicine
    med['Mechanism_of_action'] = mechanisms_dict
    
    # tally stats (if at least one ingredient was found)
    if any(value is not None for value in mechanisms_dict.values()):
        count_enriched += 1
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(medicines_data, f, indent=4, ensure_ascii=False)

print(f"\nMedicines processed: {len(medicines_data)}")
print(f"Medicines enriched: {count_enriched}")
print(f"File saved to: {output_file}")

Indexing complete.
Generics: 4247
Synonyms: 21239

Medicines processed: 11825
Medicines enriched: 10849
File saved to: processed_medicines_MoA_DrugBank.json


In [30]:
# how many drugs got matched with a MoA in DrugBank
import json
output_file= "processed_medicines_MoA_DrugBank.json"
with open(output_file, 'r', encoding='utf-8') as f:
    check_data = json.load(f)

all_ingredients = set()
matched_ingredients = set()

for med in check_data:
    moa = med.get('Mechanism_of_action', {})
    if not isinstance(moa, dict):
        continue
    for ingredient, mech in moa.items():
        key = ingredient.strip().lower()
        all_ingredients.add(key)
        if mech is not None:
            matched_ingredients.add(key)

print(f"Total unique drugs (distinct ingredients): {len(all_ingredients)}")
print(f"Unique drugs matched with a MoA: {len(matched_ingredients)}")
print(f"Unique drugs WITHOUT a MoA: {len(all_ingredients) - len(matched_ingredients)}")


Total unique drugs (distinct ingredients): 1058
Unique drugs matched with a MoA: 780
Unique drugs WITHOUT a MoA: 278


### **4.1. Tags analysis**

In [31]:
# Note: some tags are not properly closed in the source data (e.g. "K<sub>2</sub").
# Looking at the HTML tags present in the DrugBank Mechanism_of_action field:

import json
import re
import os

meds_file = "processed_medicines.json"
drugbank_file = os.path.join("..", "DrugBank", "drug_data_test_status.json")

# load the DrugBank dataset
try:
    with open(drugbank_file, 'r', encoding='utf-8') as f:
        drugbank_list = json.load(f)
    print(f"Total records loaded: {len(drugbank_list)}")
except FileNotFoundError:
    print(f"ERROR: file not found: {drugbank_file}")
    drugbank_list = []

# HTML tags
found_tags = set()
count_analyzed = 0

if drugbank_list:

    for item in drugbank_list:
        # get the mechanism-of-action text
        moa_text = item.get('Mechanism_of_action', '')
        
        # skip if there's no text or it's None
        if not moa_text:
            continue
        
        count_analyzed += 1
        
        # </?             -> starts with < or </
        # ([a-zA-Z]       -> the FIRST character HAS to be a letter (blocks <50>)
        # [a-zA-Z0-9]*)   -> the rest can be letters or digits (catches h1, h2)
        # [^>]* -> ignores attributes
        # >               -> end of the tag
        matches = re.findall(r'</?([a-zA-Z][a-zA-Z0-9]*)[^>]*>', str(moa_text))
        
        for tag in matches:
            found_tags.add(tag.lower())

    print(f"Mechanisms analyzed: {count_analyzed}")
    print("Unique tags found:")

    if found_tags:
        sorted_tags = sorted(list(found_tags))
        for tag in sorted_tags:
            print(f"  <{tag}>")
    else:
        print("No HTML tags found.")

else:
    print("The DrugBank list is empty.")

Total records loaded: 19830
Mechanisms analyzed: 4247
Unique tags found:
  <br>
  <i>
  <sub>
  <sup>


### **4.2 Cleaning MoA**

In [32]:
import json
import re
import html
import os

input_file = "processed_medicines_MoA_DrugBank.json"
output_file = "processed_medicines_vfinal.json"

def clean_moa_text(text):
    if text is None or not isinstance(text, str):
        return None
    
    # strip HTML
    text = html.unescape(text)
    text = re.sub(r'(?i)</?(?:br|i|sub|sup)\b(?:/?>)?', '', text)

    # remove references
    # only strips the content inside square brackets
    pattern = r'\[[a-zA-Z0-9\s,\.]+\]'
    text = re.sub(pattern, '', text)

    # whitespace and punctuation normalization
    
    # A. collapse multiple spaces into one
    # "Text   ,  ." -> "Text , ."
    text = re.sub(r'\s+', ' ', text).strip()

    # B. remove spaces before punctuation
    # "Text , ." -> "Text,."
    # matches a space followed by a period, comma, semicolon or colon
    text = re.sub(r'\s+([.,;:])', r'\1', text)

    # C. comma followed by a period
    # "Text,." -> "Text."
    # also handles cases like "Text,,." when there are several
    text = re.sub(r',+\.', '.', text)

    # D. if the sentence ended with a comma
    # "Text," -> "Text."
    text = re.sub(r',+$', '.', text)
    
    # E. fix a duplicated comma (can happen between refs mid-sentence)
    # "Text,, continuation" -> "Text, continuation"
    text = re.sub(r',+,', ',', text)

    # clean up quotes
    text = text.replace('"', "'")

    return text

if os.path.exists(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    count = 0
    for med in data:
        moa_dict = med.get('Mechanism_of_action', {})
        if not moa_dict: continue 

        for ingredient, raw_text in moa_dict.items():
            if raw_text:
                clean_text = clean_moa_text(raw_text)
                moa_dict[ingredient] = clean_text
                count += 1

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Processing complete.")
    print(f"Clean file saved to: {output_file}")

else:
    print("Error: file not found.")

Processing complete.
Clean file saved to: processed_medicines_vfinal.json


## **5. Cleaning up the medicines names**

### **5.1. Extracting Unique Suffixes After the Word 'For'**
- This script scans the drug database looking for the word "for" (in isolation). Its goal is to capture only the text that follows this word (for example, "injection", "oral suspension").
- The final result is saved in a JSON file containing only a clean and ordered list with all the unique suffix variations found in the dataset, facilitating analysis for data cleaning.

In [33]:
import json
import re
import os
from collections import Counter

INPUT_FILE = "processed_medicines_vfinal.json"
OUTPUT_SUFFIXES_FILE = "unique_suffixes_after_for_with_counts.json"
OUTPUT_FULL_NAMES_FILE = "medicines_full_names_with_for.json"

def analyze_for_suffixes():
    print(f"Extracting suffixes after 'for' in '{INPUT_FILE}'...")

    if not os.path.exists(INPUT_FILE):
        print(f"Error: file '{INPUT_FILE}' not found.")
        return

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Counter will automatically tally the occurrences of each suffix
        suffix_counts = Counter()
        # List to store the full original names of the medicines
        full_names_list = []

        # Regex explanation:
        # \bfor\b  -> Finds the exact word "for" (ignores words like "metformin")
        # \s* -> Catches any spaces right after "for"
        # (.*)     -> Captures absolutely everything else until the end of the string into Group 1
        pattern = re.compile(r'\bfor\b\s*(.*)', re.IGNORECASE)

        for med in data:
            name = med.get('Medicine Name', '')
            
            # Search for the pattern in the medicine name
            match = pattern.search(name)
            
            if match:
                # 1. Save the full medicine name
                full_names_list.append(name)
                
                # 2. Extract and tally the suffix
                suffix = match.group(1).strip()
                if suffix:
                    suffix_counts[suffix] += 1

        # Sort the dictionary by frequency (descending), then alphabetically as a tie-breaker
        sorted_suffixes = dict(sorted(suffix_counts.items(), key=lambda item: (-item[1], item[0].casefold())))

        # save results:
        
        # Save 1: Suffixes with their respective counts
        with open(OUTPUT_SUFFIXES_FILE, 'w', encoding='utf-8') as f_out_suffixes:
            json.dump(sorted_suffixes, f_out_suffixes, indent=4, ensure_ascii=False)

        # Save 2: Full medicine names
        with open(OUTPUT_FULL_NAMES_FILE, 'w', encoding='utf-8') as f_out_names:
            json.dump(full_names_list, f_out_names, indent=4, ensure_ascii=False)

        # --- PRINT SUMMARY ---
        print(f"Total medicines with 'for': {len(full_names_list)}")
        print(f"Unique suffixes found: {len(sorted_suffixes)}")
        print(f"Suffix counts saved to: {OUTPUT_SUFFIXES_FILE}")
        print(f"Full names saved to: {OUTPUT_FULL_NAMES_FILE}")
        
        if sorted_suffixes:
            print("\nTop most frequent suffixes:")
            # Print the top 10 most common suffixes
            for suffix, count in list(sorted_suffixes.items())[:10]:
                print(f"  {suffix}: {count} time(s)")
            if len(sorted_suffixes) > 10:
                print("  ... (check the JSON file for the complete list)")

    except json.JSONDecodeError:
        print("Error: the file is not a valid JSON.")
    except Exception as e:
        print(f"Unexpected error: {e}")

analyze_for_suffixes()

Extracting suffixes after 'for' in 'processed_medicines_vfinal.json'...
Total medicines with 'for': 61
Unique suffixes found: 16
Suffix counts saved to: unique_suffixes_after_for_with_counts.json
Full names saved to: medicines_full_names_with_for.json

Top most frequent suffixes:
  Injection: 29 time(s)
  Injection 40IU/ml: 5 time(s)
  Oral Suspension: 5 time(s)
  Infusion: 4 time(s)
  Injection 100IU/ml: 4 time(s)
  Antifungal Infections: 2 time(s)
  Oral Solution: 2 time(s)
  Skin Infections: 2 time(s)
  -IC Capsule: 1 time(s)
  Acne Treatment: 1 time(s)
  ... (check the JSON file for the complete list)


### **5.2 Selective Noise Reduction: Formulation vs. Indication**

During the data curation phase, analysis of the dataset revealed that the preposition "for" is used in two distinct contexts in drug nomenclature:
1. **Technical Formulation:** Indicating routes of administration and dosages.
2. **Therapeutic Indication:** Indicating the condition treatment.

The decision was made to perform selective noise reduction instead of indiscriminately cutting all text after the word "for".

Indiscriminate removal of formulation data would result in the critical loss of dosage information, potentially creating indistinguishable duplicates in the dataset. For example, cutting the technical suffixes from *Human Mixtard 70/30 Suspension for Injection 40IU/ml* and *Human Mixtard 70/30 Suspension for Injection 100IU/ml* would reduce both to an identical and ambiguous name.

Conversely, indication-based suffixes (e.g., *"for Acne Treatment"* or *"for Oral Care"*) shouldn't be in the name of the medicine. As such, only a specific subset of predefined marketing phrases and indications was targeted for removal, with the aim of improving structural consistency without compromising medical rigor.

In [ ]:
import json
import re
import os

INPUT_FILE = "processed_medicines_vfinal.json"
OUTPUT_FILE = "processed_medicines_vfinal.json"

# list of exact suffixes to remove ( indications)
# we only want to remove these specific phrases when they follow "for"
TARGET_SUFFIXES = [
    "Antifungal Infections",
    "Skin Infections",
    "Acne Treatment",
    "Eye Infections",
    "Fever &amp; Pain Management",  
    "Oral Care",
    "Vitamin B12 Deficiency",
    "Kidney &amp; Gout Stone"       
]

def clean_indication_suffixes():
    print(f"Selective suffix cleanup on '{INPUT_FILE}'...")

    if not os.path.exists(INPUT_FILE):
        print(f"Error: file '{INPUT_FILE}' not found.")
        return

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # build a solid regex
        escaped_targets = [re.escape(suffix) for suffix in TARGET_SUFFIXES]
        pattern_str = r'\s*\bfor\b\s*(?:' + '|'.join(escaped_targets) + r')\s*$'
        
        cleaning_regex = re.compile(pattern_str, re.IGNORECASE)
        
        cleaned_count = 0

        for med in data:
            original_name = med.get('Medicine Name', '')
            
            if original_name:
                # replace the matched pattern with an empty string
                new_name = cleaning_regex.sub('', original_name)
                
                # if the name changed, the regex found and removed the suffix
                if new_name != original_name:
                    med['Medicine Name'] = new_name
                    cleaned_count += 1

        # --- SAVE THE CLEAN DATASET ---
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
            json.dump(data, f_out, indent=4, ensure_ascii=False)

        # --- PRINT SUMMARY ---
        print(f"Total medicines processed: {len(data)}")
        print(f"Medicines cleaned: {cleaned_count}")
        print(f"Clean dataset saved to: {OUTPUT_FILE}")

    except json.JSONDecodeError:
        print("Error: the file is not a valid JSON.")
    except Exception as e:
        print(f"Unexpected error: {e}")

clean_indication_suffixes()

Selective suffix cleanup on 'processed_medicines_vfinal.json'...
Total medicines processed: 11825
Medicines cleaned: 10
Clean dataset saved to: processed_medicines_vfinal.json


## **Notes on the final dataset:**

- There are 11 medications where "Side_Effects" shows "No common side effects seen", Nausea, etc. The solution will be to remove "No common side effects seen".
- 160 - 11 = 149 medications show "No common side effects seen" as "Side_Effects".
- 98 medications show "[]" as "Side_Effects" because they previously had "Limited data available".
- Medications whose components did not have a mechanism of action in the drugbank were left as null, e.g., "Zinc pyrithione": null